# Weather Pipeline — OpenWeatherMap API




**You need:** A free API key from https://openweathermap.org/api


In [3]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [4]:
import requests
import pandas as pd
from datetime import datetime

WEATHER_API_KEY = "18890a1d8cfd1b3130af9ac578d2c1a5"  # ← paste your key


---
## Step 1: Extract weather for 40 cities


In [5]:
CITIES = [
    "Nairobi", "London", "New York", "Tokyo", "Sydney",
    "Lagos", "Dubai", "Berlin", "Paris", "Mumbai",
    "Cairo", "Toronto", "Singapore", "Bangkok", "Istanbul",
    "Moscow", "Seoul", "Jakarta", "Lima", "Bogota",
    "Johannesburg", "Accra", "Addis Ababa", "Casablanca", "Dar es Salaam",
    "Kampala", "Kigali", "Lusaka", "Maputo", "Kinshasa",
    "Sao Paulo", "Mexico City", "Buenos Aires", "Santiago", "Havana",
    "Amsterdam", "Stockholm", "Oslo", "Dublin", "Rome",
]

raw_data = []
for city in CITIES:
    r = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={"q": city, "appid": WEATHER_API_KEY, "units": "metric"},
        verify=False
    )
    if r.status_code == 200:
        raw_data.append(r.json())
    else:
        print(f"  Failed: {city} ({r.status_code})")

print(f"\nExtracted weather for {len(raw_data)} cities")



Extracted weather for 40 cities


---
## Step 2: Clean and save to CSV (Bronze)


In [6]:
rows = []
for w in raw_data:
    rows.append({
        "city":          w["name"],
        "country":       w["sys"]["country"],
        "temp_c":        round(float(w["main"]["temp"]), 1),
        "feels_like_c":  round(float(w["main"]["feels_like"]), 1),
        "humidity_pct":  int(w["main"]["humidity"]),
        "pressure_hpa":  int(w["main"]["pressure"]),
        "wind_speed_ms": round(float(w["wind"]["speed"]), 1),
        "weather":       w["weather"][0]["main"],
        "description":   w["weather"][0]["description"],
        "visibility_m":  int(w.get("visibility", 0)),
        "latitude":      round(float(w["coord"]["lat"]), 4),
        "longitude":     round(float(w["coord"]["lon"]), 4),
        "reading_time":  datetime.utcfromtimestamp(w["dt"]).strftime("%Y-%m-%d %H:%M:%S"),
        "extracted_at":  datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })

df = pd.DataFrame(rows)
df.to_csv("weather_raw.csv", index=False)
print(f"Saved {len(df)} rows to weather_raw.csv")
df.head()


Saved 40 rows to weather_raw.csv


C:\Users\simon.ngugi\AppData\Local\Temp\ipykernel_31264\4194516087.py:16: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "reading_time":  datetime.utcfromtimestamp(w["dt"]).strftime("%Y-%m-%d %H:%M:%S"),


,city,country,temp_c,feels_like_c,humidity_pct,pressure_hpa,wind_speed_ms,weather,description,visibility_m,latitude,longitude,reading_time,extracted_at
0,Nairobi,KE,18.6,18.6,77,1019,3.1,Clouds,overcast clouds,10000,-1.2833,36.8167,2026-10-19 06:08:28,2026-10-19 09:11:11
1,London,GB,9.1,8.1,92,1026,2.1,Clear,clear sky,10000,51.5085,-0.1257,2026-10-19 05:56:46,2026-10-19 09:11:11
2,New York,US,10.3,9.2,70,1033,3.6,Clear,clear sky,10000,40.7143,-74.0060,2026-10-19 06:08:50,2026-10-19 09:11:11
3,Tokyo,JP,25.8,26.0,60,1018,8.8,Clouds,overcast clouds,10000,35.6895,139.6917,2026-10-19 05:58:14,2026-10-19 09:11:11
4,Sydney,AU,22.0,21.8,57,1025,8.8,Clear,clear sky,10000,-33.8679,151.2073,2026-10-19 06:10:13,2026-10-19 09:11:11


---
## Step 3: Load raw CSV


In [7]:
df = pd.read_csv("weather_raw.csv")
print(f"Loaded {len(df)} rows from weather_raw.csv")
df.head()


Loaded 40 rows from weather_raw.csv


,city,country,temp_c,feels_like_c,humidity_pct,pressure_hpa,wind_speed_ms,weather,description,visibility_m,latitude,longitude,reading_time,extracted_at
0,Nairobi,KE,18.6,18.6,77,1019,3.1,Clouds,overcast clouds,10000,-1.2833,36.8167,2026-10-19 06:08:28,2026-10-19 09:11:11
1,London,GB,9.1,8.1,92,1026,2.1,Clear,clear sky,10000,51.5085,-0.1257,2026-10-19 05:56:46,2026-10-19 09:11:11
2,New York,US,10.3,9.2,70,1033,3.6,Clear,clear sky,10000,40.7143,-74.0060,2026-10-19 06:08:50,2026-10-19 09:11:11
3,Tokyo,JP,25.8,26.0,60,1018,8.8,Clouds,overcast clouds,10000,35.6895,139.6917,2026-10-19 05:58:14,2026-10-19 09:11:11
4,Sydney,AU,22.0,21.8,57,1025,8.8,Clear,clear sky,10000,-33.8679,151.2073,2026-10-19 06:10:13,2026-10-19 09:11:11


---
## Step 4: SCD2 — Track changes over time

**Tracked columns:** `temp_c`, `humidity_pct`, `weather`

Weather changes constantly, so every run will produce new SCD2 rows.
This is perfect for demonstrating how SCD2 builds up history.


In [8]:
import os

KEY_COL    = "city"
TRACK_COLS = ["temp_c", "humidity_pct", "weather"]
SCD2_FILE  = "weather_scd2.csv"
now        = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Load existing SCD2 or start empty
if os.path.exists(SCD2_FILE):
    scd2 = pd.read_csv(SCD2_FILE)
    print(f"Existing SCD2: {len(scd2)} rows")
else:
    scd2 = pd.DataFrame()
    print("No existing SCD2 file — starting fresh")


No existing SCD2 file — starting fresh


In [9]:
# Get current (active) rows
if len(scd2) > 0:
    current = scd2[scd2["is_current"] == True].copy()
else:
    current = pd.DataFrame()

new_rows    = []
changed_ids = []

for _, row in df.iterrows():
    city = row[KEY_COL]

    match = current[current[KEY_COL] == city] if len(current) > 0 else pd.DataFrame()

    if len(match) == 0:
        # NEW city
        new_rows.append(row)

    else:
        # EXISTING — check tracked columns
        old = match.iloc[0]
        changed = False
        for col in TRACK_COLS:
            if str(row[col]) != str(old[col]):
                changed = True
                break

        if changed:
            changed_ids.append(city)
            new_rows.append(row)

print(f"New:       {len(new_rows) - len(changed_ids)}")
print(f"Changed:   {len(changed_ids)}")
print(f"Unchanged: {len(df) - len(new_rows)}")


New:       40
Changed:   0
Unchanged: 0


In [10]:
# Expire old rows for changed cities
if len(changed_ids) > 0 and len(scd2) > 0:
    mask = (scd2[KEY_COL].isin(changed_ids)) & (scd2["is_current"] == True)
    scd2.loc[mask, "is_current"] = False
    scd2.loc[mask, "valid_to"]   = now

# Build new SCD2 rows
inserts = pd.DataFrame(new_rows)
inserts["valid_from"]  = now
inserts["valid_to"]    = "9999-12-31 00:00:00"
inserts["is_current"]  = True

# Combine and save
scd2 = pd.concat([scd2, inserts], ignore_index=True)
scd2.to_csv(SCD2_FILE, index=False)

total   = len(scd2)
active  = len(scd2[scd2["is_current"] == True])
history = total - active
print(f"\nSCD2 saved: {total} total rows ({active} current, {history} historical)")



SCD2 saved: 40 total rows (40 current, 0 historical)


---
## Verify


In [ ]:
scd2 = pd.read_csv(SCD2_FILE)

print(f"Total rows:      {len(scd2)}")
print(f"Current rows:    {len(scd2[scd2['is_current'] == True])}")
print(f"Historical rows: {len(scd2[scd2['is_current'] == False])}")
print(f"Unique cities:   {scd2['city'].nunique()}")

# Show a sample
scd2.head(10)


---
## How to see SCD2 in action

Run this notebook again in a few hours. Weather changes constantly,
so most cities will have new readings. You'll see:
- Old rows with `is_current = False` and a `valid_to` timestamp
- New rows with `is_current = True` and updated temps/humidity

Run it 3-4 times over a day and you'll have a proper history table.
